# TravelTide Mastery Project

## Datenaufbereitung und explorative Datenanalyse

Ziel des Projekts ist die Entwicklung verhaltensbasierter Kundenmerkmale, 
um Kunden passende Vorteile eines Rewards-Programms zuzuordnen.

Die Kohorte wurde vorab mit SQL definiert. Sie umfasst Nutzer mit mehr als 
sieben Sessions seit dem 04.01.2023.


In [2]:
# Importieren der Bibliotheken
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# CSV-Dateien in Dataframes umwandeln.

users = pd.read_csv("../data/users.csv")
sessions = pd.read_csv("../data/sessions.csv")
flights = pd.read_csv("../data/flights.csv")
hotels = pd.read_csv("../data/hotels.csv")

In [4]:
# Kurze Überblick über die Größe des Datensatzes.
 
print("Users:", users.shape)
print("Sessions:", sessions.shape)
print("Flights:", flights.shape)
print("Hotels:", hotels.shape)

Users: (5998, 11)
Sessions: (49211, 13)
Flights: (13717, 13)
Hotels: (14313, 7)


In [5]:
# Erste Kontrolle durch Darstellung der obersten Zeilen.

display(users.head())
display(sessions.head())
display(flights.head())
display(hotels.head())

,user_id,birthdate,gender,married,has_children,home_country,home_city,home_airport,home_airport_lat,home_airport_lon,sign_up_date
0,23557,1958-12-08,F,True,False,usa,new york,LGA,40.777,-73.872,2021-07-22
1,94883,1972-03-16,F,True,False,usa,kansas city,MCI,39.297,-94.714,2022-02-07
2,101486,1972-12-07,F,True,True,usa,tacoma,TCM,47.138,-122.476,2022-02-17
3,101961,1980-09-14,F,True,False,usa,boston,BOS,42.364,-71.005,2022-02-17
4,106907,1978-11-17,F,True,True,usa,miami,TNT,25.862,-80.897,2022-02-24


,session_id,user_id,trip_id,session_start,session_end,flight_discount,hotel_discount,flight_discount_amount,hotel_discount_amount,flight_booked,hotel_booked,page_clicks,cancellation
0,510198-48c0e1cf4fb147328bb5dbae3ca22411,510198,510198-74464b50ee984494b39460eef1e03bb2,2023-01-04 00:01:00,2023-01-04 00:04:23,False,True,NaN,0.15,True,True,28,False
1,509353-fcc7b28f87464773b8ce9a91d3b8e0df,509353,509353-f2916a48af814fd695e9617f30a86833,2023-01-04 00:52:00,2023-01-04 00:54:11,False,False,NaN,NaN,True,True,17,False
2,510571-f9265397db5e4921b1b6f569a88b2a7f,510571,510571-ddad02f4de2641c885409da85447df8d,2023-01-04 01:06:00,2023-01-04 01:07:44,True,False,0.2,NaN,False,True,14,False
3,407250-8a0a1f8f48a84e6e9d70f3582d350b43,407250,407250-9152c91fdf754cffbb345776da107919,2023-01-04 01:10:00,2023-01-04 01:12:27,False,False,NaN,NaN,True,False,20,False
4,468808-2ece675cf2f747fb8690b7c9f474c3d7,468808,NaN,2023-01-04 01:20:00,2023-01-04 01:21:08,False,False,NaN,NaN,False,False,9,False


,trip_id,origin_airport,destination,destination_airport,seats,return_flight_booked,departure_time,return_time,checked_bags,trip_airline,destination_airport_lat,destination_airport_lon,base_fare_usd
0,101486-1015905607d74b15954bfd4ac7029ef3,TCM,edmonton,YED,1,True,2023-06-10 10:00:00,2023-06-14 10:00:00,0,United Airlines,53.667,-113.467,189.91
1,101961-19f641633ebc442799662326f0b4dfa0,BOS,new york,LGA,1,True,2023-05-05 11:00:00,2023-05-09 11:00:00,0,Allegiant Air,40.640,-73.779,49.67
2,101961-29a8ff7c9910469c959fffa60215cf78,BOS,montreal,YHU,1,True,2023-02-08 07:00:00,2023-02-13 07:00:00,1,United Airlines,45.517,-73.417,77.02
3,101961-836fd88487d240baa4402c8e4c6f188c,BOS,seattle,BFI,1,True,2023-03-16 07:00:00,2023-03-21 07:00:00,1,Kenmore Air,47.530,-122.302,769.50
4,101961-c4c922fbc83342779565d249ee5e6bf7,BOS,charlotte,CLT,1,True,2023-06-27 11:00:00,2023-07-05 11:00:00,0,JetBlue Airways,35.214,-80.943,216.57


,trip_id,hotel_name,nights,rooms,check_in_time,check_out_time,hotel_per_room_usd
0,101486-1015905607d74b15954bfd4ac7029ef3,Crowne Plaza - edmonton,3,1,2023-06-10 13:12:24.03,2023-06-14 11:00:00,253
1,101486-6759c5dd49a1457d916bb2bbf48c3115,Banyan Tree - montreal,5,2,2023-06-24 11:00:00,2023-06-29 11:00:00,144
2,101961-19f641633ebc442799662326f0b4dfa0,Conrad - new york,3,1,2023-05-05 13:22:30.72,2023-05-09 11:00:00,165
3,101961-29a8ff7c9910469c959fffa60215cf78,Rosewood - montreal,4,1,2023-02-08 09:30:00.99,2023-02-12 11:00:00,197
4,101961-836fd88487d240baa4402c8e4c6f188c,Extended Stay - seattle,4,1,2023-03-16 14:00:14.67,2023-03-21 11:00:00,132


In [ ]:
# Kontrolle, ob Datumangaben korrekt aus der CSV übernommen wurden.

print(users[["birthdate", "sign_up_date"]].dtypes)
print(sessions[["session_start", "session_end"]].dtypes)
print(flights[["departure_time", "return_time"]].dtypes)
print(hotels[["check_in_time", "check_out_time"]].dtypes)

birthdate       str
sign_up_date    str
dtype: object
session_start    str
session_end      str
dtype: object
departure_time    str
return_time       str
dtype: object
check_in_time     str
check_out_time    str
dtype: object


In [7]:
# Umwandlung der Datumsangaben in das Datetime-Format.

users["birthdate"] = pd.to_datetime(
    users["birthdate"],
    format="mixed"
)

users["sign_up_date"] = pd.to_datetime(
    users["sign_up_date"],
    format="mixed"
)

sessions["session_start"] = pd.to_datetime(
    sessions["session_start"],
    format="mixed"
)

sessions["session_end"] = pd.to_datetime(
    sessions["session_end"],
    format="mixed"
)

flights["departure_time"] = pd.to_datetime(
    flights["departure_time"],
    format="mixed"
)

flights["return_time"] = pd.to_datetime(
    flights["return_time"],
    format="mixed"
)

hotels["check_in_time"] = pd.to_datetime(
    hotels["check_in_time"],
    format="mixed"
)

hotels["check_out_time"] = pd.to_datetime(
    hotels["check_out_time"],
    format="mixed"
)

In [8]:
# Kontrolle, ob die Umwandlung korrekt erfolgt ist und keine Werte verloren gegangen sind.

print(users[["birthdate", "sign_up_date"]].dtypes)
print(sessions[["session_start", "session_end"]].dtypes)
print(flights[["departure_time", "return_time"]].dtypes)
print(hotels[["check_in_time", "check_out_time"]].dtypes)

print("\nFehlende session_start:", sessions["session_start"].isna().sum())
print("Fehlende session_end:", sessions["session_end"].isna().sum())
print("Fehlende departure_time:", flights["departure_time"].isna().sum())
print("Fehlende return_time:", flights["return_time"].isna().sum())
print("Fehlende check_in_time:", hotels["check_in_time"].isna().sum())
print("Fehlende check_out_time:", hotels["check_out_time"].isna().sum())

birthdate       datetime64[us]
sign_up_date    datetime64[us]
dtype: object
session_start    datetime64[us]
session_end      datetime64[us]
dtype: object
departure_time    datetime64[us]
return_time       datetime64[us]
dtype: object
check_in_time     datetime64[us]
check_out_time    datetime64[us]
dtype: object

Fehlende session_start: 0
Fehlende session_end: 0
Fehlende departure_time: 0
Fehlende return_time: 597
Fehlende check_in_time: 0
Fehlende check_out_time: 0


## 2. Datenqualitätsprüfung

In diesem Abschnitt werden die vier Ausgangstabellen auf ihre Struktur,
fehlende Werte, Duplikate, Schlüsselbeziehungen und unplausible Werte geprüft.
Die Rohdaten werden dabei noch nicht verändert.

In [9]:
# Tabellenüberischt erstellen.

dataframes = {
    "users": users,
    "sessions": sessions,
    "flights": flights,
    "hotels": hotels
}

overview = pd.DataFrame({
    "rows": {name: len(df) for name, df in dataframes.items()},
    "columns": {name: df.shape[1] for name, df in dataframes.items()},
    "duplicate_rows": {
        name: df.duplicated().sum()
        for name, df in dataframes.items()
    },
    "missing_values": {
        name: df.isna().sum().sum()
        for name, df in dataframes.items()
    }
})

overview

,rows,columns,duplicate_rows,missing_values
users,5998,11,0,0
sessions,49211,13,0,116444
flights,13717,13,0,597
hotels,14313,7,0,0


In [10]:
# Datentypen kontrollieren

for name, df in dataframes.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)


USERS
user_id                      int64
birthdate           datetime64[us]
gender                         str
married                       bool
has_children                  bool
home_country                   str
home_city                      str
home_airport                   str
home_airport_lat           float64
home_airport_lon           float64
sign_up_date        datetime64[us]
dtype: object

SESSIONS
session_id                           str
user_id                            int64
trip_id                              str
session_start             datetime64[us]
session_end               datetime64[us]
flight_discount                     bool
hotel_discount                      bool
flight_discount_amount           float64
hotel_discount_amount            float64
flight_booked                       bool
hotel_booked                        bool
page_clicks                        int64
cancellation                        bool
dtype: object

FLIGHTS
trip_id                     

In [11]:
# Fehlende Werte untersuchen.

for name, df in dataframes.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name.upper()}")

    if missing.empty:
        print("Keine fehlenden Werte.")
    else:
        print(missing)


USERS
Keine fehlenden Werte.

SESSIONS
trip_id                   32509
flight_discount_amount    40929
hotel_discount_amount     43006
dtype: int64

FLIGHTS
return_time    597
dtype: int64

HOTELS
Keine fehlenden Werte.


In [12]:
# Vollständig doppelte Zeilen prüfen.

for name, df in dataframes.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name}: {duplicate_count} vollständig doppelte Zeilen")

users: 0 vollständig doppelte Zeilen
sessions: 0 vollständig doppelte Zeilen
flights: 0 vollständig doppelte Zeilen
hotels: 0 vollständig doppelte Zeilen


In [13]:
# Primärschlüssel prüfen

key_checks = pd.DataFrame({
    "table": ["users", "sessions", "flights", "hotels"],
    "key": ["user_id", "session_id", "trip_id", "trip_id"],
    "rows": [
        len(users),
        len(sessions),
        len(flights),
        len(hotels)
    ],
    "unique_keys": [
        users["user_id"].nunique(),
        sessions["session_id"].nunique(),
        flights["trip_id"].nunique(),
        hotels["trip_id"].nunique()
    ],
    "missing_keys": [
        users["user_id"].isna().sum(),
        sessions["session_id"].isna().sum(),
        flights["trip_id"].isna().sum(),
        hotels["trip_id"].isna().sum()
    ]
})

key_checks["key_is_unique"] = (
    key_checks["rows"] == key_checks["unique_keys"]
)

key_checks

,table,key,rows,unique_keys,missing_keys,key_is_unique
0,users,user_id,5998,5998,0,True
1,sessions,session_id,49211,49211,0,True
2,flights,trip_id,13717,13717,0,True
3,hotels,trip_id,14313,14313,0,True


In [14]:
# Beziehungen zwischen den Tabellen prüfen

session_user_ids = set(sessions["user_id"])
user_ids = set(users["user_id"])

session_trip_ids = set(sessions["trip_id"].dropna())
flight_trip_ids = set(flights["trip_id"])
hotel_trip_ids = set(hotels["trip_id"])

print(
    "Session-Nutzer ohne passenden Eintrag in users:",
    len(session_user_ids - user_ids)
)

print(
    "Flüge ohne passende trip_id in sessions:",
    len(flight_trip_ids - session_trip_ids)
)

print(
    "Hotels ohne passende trip_id in sessions:",
    len(hotel_trip_ids - session_trip_ids)
)

Session-Nutzer ohne passenden Eintrag in users: 0
Flüge ohne passende trip_id in sessions: 0
Hotels ohne passende trip_id in sessions: 0


In [ ]:
# Sessions pro User prüfen (laut Kohortendefinition mindestens 7 Sessions pro User).

sessions_per_user = (
    sessions.groupby("user_id")["session_id"]
    .nunique()
)

sessions_per_user.describe()

print("Minimum:", sessions_per_user.min())
print("Maximum:", sessions_per_user.max())
print(
    "Nutzer mit höchstens 7 Sessions:",
    (sessions_per_user <= 7).sum()
)

Minimum: 8
Maximum: 12
Nutzer mit höchstens 7 Sessions: 0


## Zusammenfassung der Datenqualitätsprüfung

Die vier Ausgangstabellen `users`, `sessions`, `flights` und `hotels` wurden hinsichtlich ihrer Struktur, Vollständigkeit und Konsistenz überprüft.

* **Tabellenstruktur:** Die Anzahl der Zeilen und Spalten entspricht den erwarteten Exportergebnissen der definierten Kundenkohorte.
* **Datentypen:** IDs, kategoriale Merkmale, numerische Werte, Boolean-Variablen und Zeitangaben wurden mit passenden Datentypen eingelesen. Eine weitere Typkonvertierung ist derzeit nicht erforderlich.
* **Primärschlüssel:** Die Primärschlüssel `user_id`, `session_id` und `trip_id` sind in den jeweiligen Tabellen vollständig vorhanden und eindeutig.
* **Tabellenbeziehungen:** Alle Nutzer-IDs aus `sessions` besitzen einen passenden Eintrag in `users`. Ebenso lassen sich die Flug- und Hotelbuchungen über `trip_id` einer Session zuordnen.
* **Kohortendefinition:** Alle enthaltenen Nutzer besitzen mehr als sieben Sessions seit dem 04.01.2023 und erfüllen damit die definierte Kohortenbedingung.
* **Fehlende Werte:** Die vorhandenen Nullwerte sind fachlich plausibel. Fehlende `trip_id`-Werte kennzeichnen Sessions ohne Buchung. Fehlende Rabattbeträge treten auf, wenn kein Rabatt angeboten wurde. Fehlende Rückflugzeiten sind bei Buchungen ohne Rückflug zu erwarten.
* **Duplikate:** In keiner der vier Tabellen wurden vollständig doppelte Zeilen gefunden.
* **Schlüsselduplikate:** Die jeweiligen Primärschlüssel sind eindeutig. Mehrfach vorkommende `trip_id`-Werte in der Session-Tabelle werden separat untersucht, da mehrere Sessions mit derselben Reise zusammenhängen können.

Die grundlegende technische Datenqualität ist damit ausreichend für die weitere Analyse. Im nächsten Schritt werden fachliche Auffälligkeiten und unplausible Werte untersucht, bevor die Tabellen zu einem gemeinsamen Session-Datensatz zusammengeführt werden.

## 3. Prüfung fachlicher Auffälligkeiten

### 3.1 Hotelübernachtungen

Die Variable `nights` sollte grundsätzlich eine positive Anzahl gebuchter
Hotelübernachtungen enthalten. Werte von null oder kleiner werden daher näher
untersucht. Mithilfe von Check-in und Check-out wird geprüft, ob sich eine
plausible Aufenthaltsdauer rekonstruieren lässt.

In [16]:
# Verteilung von nights

hotels["nights"].describe()

hotels["nights"].value_counts().sort_index()

nights
-2        1
-1      104
 0     1208
 1     3132
 2     2793
 3     1956
 4     1360
 5      888
 6      669
 7      470
 8      386
 9      286
 10     233
 11     192
 12     142
 13     111
 14      78
 15      71
 16      47
 17      42
 18      29
 19      14
 20      21
 21      20
 22      11
 23       8
 24       4
 25       7
 26       3
 27       5
 28       3
 29       4
 30       2
 31       2
 32       3
 33       1
 34       3
 35       1
 40       1
 42       1
 43       1
Name: count, dtype: int64

In [ ]:
# Ungültige bzw. auffällige Werte zählen und die Verteilung anzeigen.

invalid_nights = hotels[hotels["nights"] <= 0].copy()

print("Hotelbuchungen insgesamt:", len(hotels))
print("Buchungen mit nights <= 0:", len(invalid_nights))
print(
    "Anteil auffälliger Buchungen:",
    round(len(invalid_nights) / len(hotels) * 100, 2),
    "%"
)

invalid_nights["nights"].value_counts().sort_index()

Hotelbuchungen insgesamt: 14313
Buchungen mit nights <= 0: 1313
Anteil auffälliger Buchungen: 9.17 %


nights
-2       1
-1     104
 0    1208
Name: count, dtype: int64

Die auffälligen Werte werden zunächst nicht verändert. Vor einer Bereinigung
wird geprüft, ob die Aufenthaltsdauer anhand von Check-in und Check-out
zuverlässig rekonstruiert werden kann. Dadurch soll vermieden werden, gültige
Buchungen voreilig zu entfernen oder unplausible Werte ohne ausreichende
Begründung zu ersetzen.

In [20]:
# Aufenthaltsdauer anhand der Zeitangaben berechnen und auffällige Fälle anzeigen.

hotels_check = hotels.copy()

hotels_check["stay_duration_hours"] = (
    hotels_check["check_out_time"] -
    hotels_check["check_in_time"]
).dt.total_seconds() / 3600

hotels_check["calendar_nights"] = (
    hotels_check["check_out_time"].dt.normalize() -
    hotels_check["check_in_time"].dt.normalize()
).dt.days

hotels_check.loc[
    hotels_check["nights"] <= 0,
    [
        "trip_id",
        "nights",
        "check_in_time",
        "check_out_time",
        "stay_duration_hours",
        "calendar_nights"
    ]
].head(20)

,trip_id,nights,check_in_time,check_out_time,stay_duration_hours,calendar_nights
16,149058-0562d645484d450b8908ae40825aaf46,0,2023-01-09 15:38:38.175,2023-01-10 11:00:00,19.356063,1
42,190866-39f7fdf3561541bd990b5bca9c8c18d1,0,2023-03-28 15:57:01.260,2023-03-29 11:00:00,19.049650,1
47,204943-104f0428d48b41548c8c62045d2b3af7,0,2023-03-28 15:40:43.050,2023-03-29 11:00:00,19.321375,1
58,206011-1eee865398c846379976aa1a8f76608f,0,2023-05-09 12:58:05.925,2023-05-10 11:00:00,22.031688,1
67,224996-c83b5f33943b4cbf8ec93049c6fb6166,0,2023-05-20 18:12:04.095,2023-05-21 11:00:00,16.798862,1
78,229330-d4a91967362b4441913d13f5bc8156a3,0,2023-01-29 11:40:26.985,2023-01-29 11:00:00,-0.674162,0
96,290123-7cf9274ec6d24ec99f26586bf5afac14,0,2023-01-31 11:52:12.585,2023-01-31 11:00:00,-0.870163,0
98,290123-929ee123c3bd48c2870569e824a22d8b,0,2023-07-19 13:46:35.715,2023-07-20 11:00:00,21.223413,1
110,312488-373f10484f764569a69d49d632dbbeaf,0,2023-07-16 11:49:54.570,2023-07-16 11:00:00,-0.831825,0
114,313724-5ff715143f1d48248171238f11446d82,0,2023-01-11 12:28:03.090,2023-01-12 11:00:00,22.532475,1


In [21]:
# Eingetragenen Wert mit Kalenderdifferenz vergleichen

pd.crosstab(
    hotels_check.loc[
        hotels_check["nights"] <= 0,
        "nights"
    ],
    hotels_check.loc[
        hotels_check["nights"] <= 0,
        "calendar_nights"
    ]
)

calendar_nights,-1,0,1
nights,,,
-2,1,0,0
-1,26,78,0
0,0,325,883


In [22]:
# Check-out vor Check-in prüfen

checkout_before_checkin = hotels_check[
    hotels_check["check_out_time"] <
    hotels_check["check_in_time"]
]

print(
    "Buchungen mit Check-out vor Check-in:",
    len(checkout_before_checkin)
)

Buchungen mit Check-out vor Check-in: 226


### Bereinigung der Hotelnächte

Insgesamt wurden 1.313 Hotelbuchungen mit einer Anzahl von null oder
weniger Nächten identifiziert. Dies entspricht 9,17 % der Hotelbuchungen.

Von den 1.208 Buchungen mit `nights = 0` liegen bei 883 Buchungen Check-in
und Check-out an unterschiedlichen Kalendertagen. Diese Fälle stellen
inhaltlich einen Aufenthalt mit einer Übernachtung dar. Die Nullwerte sind
vermutlich durch eine Abrundung der exakten Aufenthaltsdauer entstanden,
wenn zwischen Check-in und Check-out weniger als 24 Stunden lagen.

Da eine vorhandene Hotelbuchung mindestens eine Übernachtung umfasst,
werden alle Werte mit `nights <= 0` in der bereinigten Variable
`nights_clean` durch den Wert 1 ersetzt. Die ursprünglichen Werte werden
zur Nachvollziehbarkeit in `nights_original` erhalten.

Zusätzlich wurden 226 Buchungen identifiziert, bei denen der Check-out
zeitlich vor dem Check-in liegt. Diese Buchungen wurden nicht entfernt,
da weitere relevante Buchungsinformationen vorhanden sind. Sie wurden
mit `invalid_stay_timing` markiert. Für spätere Berechnungen der
Aufenthaltsdauer und Hotelkosten wird die bereinigte Variable
`nights_clean` verwendet.

In [ ]:
# Bereinigung der Hotelnächte in neuer Variable nights_clean, die mindestens 1 Nacht enthält.

hotels["nights_original"] = hotels["nights"]

hotels["nights_clean"] = hotels["nights"].clip(lower=1)

print(
    "Werte in nights_clean kleiner oder gleich 0 nach Bereinigung:",
    (hotels["nights_clean"] <= 0).sum()
)

Werte in nights_clean kleiner oder gleich 0 nach Bereinigung: 0


In [25]:
# Markierung von Buchungen bei denen Check-Out vor Check-In liegt

hotels["invalid_stay_timing"] = (
    hotels["check_out_time"] < hotels["check_in_time"]
)

hotels["invalid_stay_timing"].value_counts()

invalid_stay_timing
False    14087
True       226
Name: count, dtype: int64